# The Rank of a Matrix

*Course notes for **Math for Machine Learning**, C1 · W2 · L2 — "The Rank of a Matrix" (DeepLearning.AI).*

**Rank** measures how much *independent information* a matrix carries. It is the single number behind singularity, the size of a system's solution set, and applications like **image compression** (a high-rank image approximated by a low-rank one).

## Information analogy

Two sentences can carry different amounts of information:

| System | Sentences | Information |
|--------|-----------|-------------|
| 1 | "The dog is black." / "The cat is orange." | **2** pieces |
| 2 | "The dog is black." / "The dog is black." | **1** piece (repeat) |
| 3 | "The dog." / "The dog." | **0** pieces (no info) |

The **rank** of the corresponding matrix counts exactly those independent pieces of information.

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## 1. Rank of a system

Take each system with constants set to 0 (so we study the coefficients) and count independent equations:

$$
\textbf{1}\;\begin{cases}a+b=0\\a+2b=0\end{cases}\!\to\!\begin{bmatrix}1&1\\1&2\end{bmatrix}
\quad
\textbf{2}\;\begin{cases}a+b=0\\2a+2b=0\end{cases}\!\to\!\begin{bmatrix}1&1\\2&2\end{bmatrix}
\quad
\textbf{3}\;\begin{cases}0a+0b=0\\0a+0b=0\end{cases}\!\to\!\begin{bmatrix}0&0\\0&0\end{bmatrix}
$$

- System 1 — two independent equations → **rank 2**.
- System 2 — second equation repeats the first → **rank 1**.
- System 3 — no information at all → **rank 0**.

The **rank** is the number of independent rows (equivalently, the number of pivots in the row echelon form).

In [2]:
systems = {
    "System 1": [[1, 1], [1, 2]],
    "System 2": [[1, 1], [2, 2]],
    "System 3": [[0, 0], [0, 0]],
}
for name, A in systems.items():
    print(f"{name}: rank = {np.linalg.matrix_rank(A)}")

System 1: rank = 2
System 2: rank = 1
System 3: rank = 0


## 2. Rank and the dimension of the solution space

For the homogeneous system $A\mathbf{x} = \mathbf{0}$ (constants = 0), the set of solutions is a *space*, and its dimension shrinks as rank grows:

| Matrix | Rank | Solution space | Dimension |
|--------|------|----------------|-----------|
| $\begin{bmatrix}1&1\\1&2\end{bmatrix}$ | 2 | just the origin | 0 |
| $\begin{bmatrix}1&1\\2&2\end{bmatrix}$ | 1 | a line | 1 |
| $\begin{bmatrix}0&0\\0&0\end{bmatrix}$ | 0 | the whole plane | 2 |

With $n$ variables, these obey the relation

$$ \boxed{\;\text{rank} = n - \dim(\text{solution space})\;} $$

(the rank–nullity theorem — the dimension of the solution space is the **nullity**).

In [3]:
n = 2  # number of variables
for name, A in systems.items():
    rank = np.linalg.matrix_rank(A)
    nullity = n - rank          # dimension of the solution space
    print(f"{name}: rank={rank}, solution-space dimension = n - rank = {nullity}")

System 1: rank=2, solution-space dimension = n - rank = 0
System 2: rank=1, solution-space dimension = n - rank = 1
System 3: rank=0, solution-space dimension = n - rank = 2


## 3. Rank and singularity

A square $n \times n$ matrix is **non-singular** exactly when it has **full rank** ($\text{rank} = n$): every row is an independent piece of information, the solution space is just the origin, and the system has a unique solution.

- $\text{rank} = n$ → **non-singular**,
- $\text{rank} < n$ → **singular**.

So rank 2 is non-singular here, while ranks 1 and 0 are singular.

In [4]:
for name, A in systems.items():
    rank = np.linalg.matrix_rank(A)
    print(f"{name}: rank={rank}  ->  {'non-singular' if rank == n else 'singular'}")

System 1: rank=2  ->  non-singular
System 2: rank=1  ->  singular
System 3: rank=0  ->  singular


## 4. Exercise

Determine the rank of each matrix (recall $\text{rank} = 2 - \dim(\text{solution space})$):

$$ M_1 = \begin{bmatrix} 5 & 1 \\ -1 & 3 \end{bmatrix} \qquad M_2 = \begin{bmatrix} 2 & -1 \\ -6 & 3 \end{bmatrix} $$

$M_1$'s solution space has dimension 0 → **rank 2**. $M_2$'s solution space has dimension 1 → **rank 1**.

In [5]:
for name, A in {"M1": [[5, 1], [-1, 3]], "M2": [[2, -1], [-6, 3]]}.items():
    rank = np.linalg.matrix_rank(A)
    print(f"{name}: rank={rank}, solution-space dim={2 - rank}  ->  {'non-singular' if rank == 2 else 'singular'}")

M1: rank=2, solution-space dim=0  ->  non-singular
M2: rank=1, solution-space dim=1  ->  singular


## 5. Application: rank and image compression

An image is a matrix of pixel values. A detailed photo has high rank; we can **approximate** it with a low-rank matrix that needs far less data to store — this is the idea behind compression. Keeping only the largest singular values (via the SVD) lowers the rank while preserving most of the picture.

In [6]:
# Tiny illustration: a full-rank matrix vs. a rank-1 approximation from its top singular value.
rng = np.random.default_rng(0)
M = rng.integers(0, 9, size=(5, 5)).astype(float)
U, s, Vt = np.linalg.svd(M)
M_rank1 = s[0] * np.outer(U[:, 0], Vt[0])   # keep only the largest singular value

print("full matrix rank :", np.linalg.matrix_rank(M))
print("approximation rank:", np.linalg.matrix_rank(M_rank1))
print("share of 'energy' kept by the top singular value: "
      f"{s[0]**2 / np.sum(s**2):.0%}")

full matrix rank : 5
approximation rank: 1
share of 'energy' kept by the top singular value: 85%


## 6. Takeaways

- **Rank** = number of independent rows = independent "pieces of information" in a matrix (= number of pivots in REF).
- For $A\mathbf{x}=\mathbf{0}$ with $n$ variables: $\text{rank} = n - \dim(\text{solution space})$ — more rank means a smaller solution space.
- A square matrix is **non-singular ⇔ full rank** ($\text{rank}=n$); lower rank ⇒ **singular**.
- Low-rank approximation underlies **image (and data) compression**.

*Companion:* [linear dependence and independence](./C1_W1_L1_linear_dependence_and_independence.ipynb) (independent rows) · [systems to matrices / REF](./C1_W2_L1_systems_to_matrices.ipynb) (rank = number of pivots).